# 09 · 手撸 50 行 Mini Vector DB

> **学习目标**：把向量库这个「看起来神秘」的东西拆掉。50 行 NumPy 实现 add / query / delete / persist，然后用 chromadb 对照看「真正的库」多干了什么。
>
> **预备**：02 + 04 已过。
>
> **为什么重要**：你后面所有 RAG 项目都依赖向量库。知道里面是什么之后，选型 / 调参 / 排错都会更有底。

**注**：原本路线想让你「读 easy-vecdb 源码」，但本机那个目录是空的 git clone（详见 memory）。直接自己写一个反而更深入。

In [ ]:
import numpy as np
import pickle, json, time
from pathlib import Path
np.set_printoptions(precision=4, suppress=True)

## 1. 一个向量库到底要做什么

**最小三件事**：
1. `add(id, vector, metadata)` —— 存
2. `query(vector, top_k)` —— 找最相似的 k 个
3. `persist()` / `load()` —— 持久化

**生产库多干的事**：增量索引、metadata 过滤、并发安全、ANN 加速（HNSW / IVF）、磁盘块管理……我们先把骨架搞清，再聊为什么需要那些扩展。

In [ ]:
class MiniVectorDB:
    """50 行内的 brute-force 向量库。归一化后用点积 = 余弦相似度。"""

    def __init__(self, dim: int):
        self.dim = dim
        self.ids = []                                       # list[str]
        self.vectors = np.zeros((0, dim), dtype=np.float32)  # (N, dim)
        self.metadatas = []                                  # list[dict]

    @staticmethod
    def _normalize(v: np.ndarray) -> np.ndarray:
        # 单向量或一批：都按最后一维归一
        norm = np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12
        return v / norm

    def add(self, id: str, vector, metadata: dict | None = None):
        v = np.asarray(vector, dtype=np.float32).reshape(1, -1)
        assert v.shape[1] == self.dim, f'维度不匹配 {v.shape[1]} vs {self.dim}'
        v = self._normalize(v)                              # 写入时就归一，查询时不用再做
        self.ids.append(id)
        self.vectors = np.vstack([self.vectors, v])
        self.metadatas.append(metadata or {})

    def delete(self, id: str):
        idx = self.ids.index(id)
        del self.ids[idx]
        self.vectors = np.delete(self.vectors, idx, axis=0)
        del self.metadatas[idx]

    def query(self, vector, top_k: int = 3, where: dict | None = None):
        if len(self.ids) == 0:
            return []
        q = self._normalize(np.asarray(vector, dtype=np.float32).reshape(1, -1))
        sims = (self.vectors @ q.T).ravel()                # (N,)
        # 简易 metadata 过滤：所有 key/value 完全相等
        if where:
            mask = np.array([all(m.get(k) == v for k, v in where.items())
                              for m in self.metadatas])
            sims = np.where(mask, sims, -np.inf)
        order = np.argsort(-sims)[:top_k]
        return [(self.ids[i], float(sims[i]), self.metadatas[i]) for i in order if sims[i] > -np.inf]

    def persist(self, path: str):
        Path(path).write_bytes(pickle.dumps({'dim': self.dim, 'ids': self.ids,
                                              'vectors': self.vectors, 'metadatas': self.metadatas}))

    @classmethod
    def load(cls, path: str):
        d = pickle.loads(Path(path).read_bytes())
        db = cls(d['dim'])
        db.ids, db.vectors, db.metadatas = d['ids'], d['vectors'], d['metadatas']
        return db

## 2. 用它做一个最小 RAG-style demo

我们没有真实 embedding 模型（rag env 的 sentence-transformers 用不了，详见 memory），用**随机向量当 embedding** 演示流程。等学完 01-RAG 章节，你会用真实 Ollama / bge 替掉这一步，**剩下的代码完全不变**。

In [ ]:
rng = np.random.default_rng(0)

def fake_embed(text: str, dim: int = 8) -> np.ndarray:
    """假 embedding：基于文本 hash 的确定性随机向量。同样文本永远同样向量。"""
    seed = abs(hash(text)) % (2**31)
    return np.random.default_rng(seed).standard_normal(dim)

DIM = 8
db = MiniVectorDB(dim=DIM)

docs = [
    ('d1', 'Transformer 是 2017 年提出的注意力模型架构', {'topic': 'arch'}),
    ('d2', 'RAG 把检索和生成结合起来回答问题',           {'topic': 'rag'}),
    ('d3', 'LoRA 是一种参数高效的微调方法',                {'topic': 'finetune'}),
    ('d4', '向量库通常用余弦相似度做相似检索',             {'topic': 'rag'}),
    ('d5', 'Chinchilla 论文给出了参数量与数据量的 scaling law', {'topic': 'theory'}),
]
for did, text, meta in docs:
    db.add(did, fake_embed(text, DIM), {**meta, 'text': text})

print(f'库里现在有 {len(db.ids)} 条')
print('vectors shape:', db.vectors.shape)

In [ ]:
# 查询：要求和 d4 同向（因为 fake_embed 是确定性的，d4 自己最相似）
q = fake_embed('向量库通常用余弦相似度做相似检索', DIM)
print('全库 top-3:')
for did, sim, meta in db.query(q, top_k=3):
    print(f'  {did}  sim={sim:+.4f}   {meta["text"]!r}')

print('\n带 metadata 过滤 (topic=rag) 的 top-3:')
for did, sim, meta in db.query(q, top_k=3, where={'topic': 'rag'}):
    print(f'  {did}  sim={sim:+.4f}   {meta["text"]!r}')

In [ ]:
# 删 + 持久化 + 重载
db.delete('d3')
print('删 d3 后剩:', db.ids)

db.persist('./_mini_vecdb.pkl')

db2 = MiniVectorDB.load('./_mini_vecdb.pkl')
print('重载后:', db2.ids, '  vectors shape:', db2.vectors.shape)

## 3. 与 chromadb 对比 — 真库多干了什么

chromadb 1.5 在你机器的 `rag` env 里已经装好。下面跑同样的流程，**接口几乎一致**，但 chroma 多了：
- 持久化到 SQLite + Parquet（不是 pickle）
- HNSW 索引（默认 ANN，O(log N) 而不是 O(N)）
- 多 collection / 元数据 schema / 并发安全

In [ ]:
import chromadb
from chromadb.config import Settings

# 持久化到本目录
client = chromadb.PersistentClient(path='./_chroma_db')

# 干净起步（演示用）
for c in client.list_collections():
    client.delete_collection(c.name)

col = client.create_collection('demo', metadata={'hnsw:space': 'cosine'})

col.add(
    ids=[d[0] for d in docs],
    embeddings=[fake_embed(d[1], DIM).tolist() for d in docs],
    metadatas=[{'topic': d[2]['topic'], 'text': d[1]} for d in docs],
)
print('chroma 现在条数:', col.count())

In [ ]:
q_vec = fake_embed('向量库通常用余弦相似度做相似检索', DIM).tolist()
res = col.query(query_embeddings=[q_vec], n_results=3)

print('chroma top-3:')
for did, dist, meta in zip(res['ids'][0], res['distances'][0], res['metadatas'][0]):
    # chroma 返回的是 distance（越小越近）；cosine 时 distance = 1 - cosine_sim
    print(f'  {did}  distance={dist:+.4f}   sim≈{1-dist:+.4f}   {meta["text"]!r}')

print('\n→ 和我们 MiniVectorDB 的 top-3 顺序应当一致 (因为数据小、HNSW 还没退化)。')

In [ ]:
# 性能差距：10k 向量
N, D = 10_000, 128
rng = np.random.default_rng(0)
X = rng.standard_normal((N, D)).astype(np.float32)
q = rng.standard_normal(D).astype(np.float32)

# Mini DB
mini = MiniVectorDB(D)
for i in range(N):
    mini.add(f'id{i}', X[i])

t0 = time.perf_counter()
for _ in range(20):
    _ = mini.query(q, top_k=5)
mini_ms = (time.perf_counter() - t0) / 20 * 1000

# Chroma —— add 一次最多 5461 条，分批喂
col2 = client.get_or_create_collection('bench', metadata={'hnsw:space': 'cosine'})
if col2.count() == 0:
    BATCH = 4000
    for s in range(0, N, BATCH):
        e = min(s + BATCH, N)
        col2.add(ids=[f'id{i}' for i in range(s, e)], embeddings=X[s:e].tolist())

t0 = time.perf_counter()
for _ in range(20):
    _ = col2.query(query_embeddings=[q.tolist()], n_results=5)
chroma_ms = (time.perf_counter() - t0) / 20 * 1000

print(f'10k 向量 × 20 次查询，单次平均:')
print(f'  MiniDB (brute) : {mini_ms:6.2f} ms')
print(f'  Chroma (HNSW)  : {chroma_ms:6.2f} ms')
print(f'  加速比         : {mini_ms / chroma_ms:5.1f}x')
print('\n→ N 越大 HNSW 优势越夸张。100 万级以上 brute-force 基本没法用。')

## 深入思考

1. **为什么写入时归一化、查询时只点积？**
   - cosine = `q·d / (||q||·||d||)`。q 与所有 d 比较时，每个 `||d||` 都一样、可在写入时提前做。**少一次除法 × N，省下来的全是吞吐**。
2. **MiniDB 的 `delete` 用了 `np.delete`，N 大了为什么慢？**
   - `np.delete` 是 O(N) 内存复制。生产库通常用「软删除标记 + 定期 compaction」。
3. **ANN（HNSW）是怎么把 O(N) 变成 O(log N) 的？**
   - 用「分层小世界图」：高层稀疏跳大步、低层稠密微调。本质是「近似最近邻」，会有 recall 折损（通常 95%+ 可接受）。
4. **metadata 过滤要不要建索引？**
   - 我们的实现是 O(N) 扫一遍。生产库会给 metadata 字段建倒排或位图索引。Chroma / Qdrant / Weaviate 都做了。
5. **什么时候 brute-force 反而更好？**
   - N < 10k 时，brute-force 简单可控，召回 100%，且没有 ANN 的训练 / 索引构建延迟。**别拿牛刀杀鸡**。

改一改：把 MiniVectorDB 的 `_normalize` 注释掉（不归一化），看查询结果与之前是否有差异（**会差很多**，因为 fake_embed 出的向量范数差异大）。

## 自检 ✅

- [ ] 不看代码，30 分钟内写一个 brute-force 的 MiniVectorDB（含 add/query/delete/persist）。
- [ ] 解释「写入归一化、查询用点积」为什么效率更高。
- [ ] 解释 cosine distance 和 cosine similarity 的关系（`distance = 1 - sim`）。
- [ ] 解释「HNSW 为什么比 brute-force 快、付出的代价是什么」。
- [ ] 给一个 N=500 的场景，能立刻判断「该不该上 HNSW」（答：不该，brute 足够）。

## 清理（可选）

```python
import shutil
Path('./_mini_vecdb.pkl').unlink(missing_ok=True)
shutil.rmtree('./_chroma_db', ignore_errors=True)
```

## 下一步

→ 回到 [`../../README.md`](../../README.md) 把对应 checkbox 打勾，然后做一次「能力检验」。
→ 准备进入 [01-RAG](../../../01-RAG/) 章节。